In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║   CONTEXT-AWARE SARCASM DETECTION  v2.0  — COMPLETE REWRITE                  ║
# ║                                                                              ║
# ║   Model     : DeBERTa-v3-base  (≥3% F1 over RoBERTa on sarcasm tasks)        ║
# ║   Datasets  : Reddit SARC (danofer/kaggle) + TweetEval + Headlines           ║
# ║   Context   : Proper left-truncated pair encoding                            ║
# ║   Training  : Label smoothing · AdamW · cosine LR · early stopping           ║
# ║   Explain   : LIME + SHAP (token-level)                                      ║
# ║   UI        : Gradio 4 — redesigned editorial dark theme                     ║
# ║                                                                              ║
# ║   HOW TO RUN:                                                                ║
# ║     1. Runtime → Change runtime type → T4 GPU                                ║
# ║     2. (Optional) Tools → Secrets → add KAGGLE_USERNAME & KAGGLE_KEY         ║
# ║     3. Paste ENTIRE file into one cell → Run                                 ║
# ║   Est. time: ~25–35 min (GPU)  |  ~90 min (CPU, auto-trims data)             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝


# ═══════════════════════════════════════════════════════════════════════════════
# 0.  CUDA SAFETY
#
#   ⚠️  IF YOU SEE "device-side assert triggered":
#       → Runtime → Restart runtime  (Ctrl+M .)
#       → Then re-run this cell from the top.
#       A poisoned CUDA context cannot be recovered within the same runtime.
#
#   These env vars make future CUDA errors show the correct line number:
# ═══════════════════════════════════════════════════════════════════════════════
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  INSTALL
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys

PKGS = [
    "transformers>=4.40",
    "datasets",
    "scikit-learn",
    "lime",
    "gradio>=4.0",
    "plotly",
    "kaggle",
    "accelerate",
]
for pkg in PKGS:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os, warnings, gc, re, json, io, csv
warnings.filterwarnings("ignore")

import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from torch.nn import CrossEntropyLoss
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)
from lime.lime_text import LimeTextExplainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports ready  |  Device: {device}")

# ── GPU info (no-op reset — a poisoned context requires Runtime → Restart) ──
if device.type == "cuda":
    print(f"✅ GPU ready  |  {torch.cuda.get_device_name(0)}  |  "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB total")
    torch.cuda.empty_cache()


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  CONFIGURATION  (edit these to taste)
# ═══════════════════════════════════════════════════════════════════════════════
# ── Model choice ──────────────────────────────────────────────────────────────
# roberta-base  → 125M params, ~4-5 it/s on T4, ~25 min total. RECOMMENDED.
# deberta-v3-base → 184M params, ~0.6 it/s on T4, ~3.5 hrs. Use on A100 only.
MODEL_CHOICE = "roberta-base"   # ← change to "microsoft/deberta-v3-base" if on A100

CFG = dict(
    model_name   = MODEL_CHOICE,
    max_len      = 128,
    epochs       = 3,
    batch_gpu    = 32,            # RoBERTa fits batch=32 easily on T4
    grad_accum   = 1,             # no accumulation needed — batch=32 is already efficient
    batch_cpu    = 8,
    lr           = 2e-5,          # well-established for RoBERTa fine-tuning
    weight_decay = 0.01,
    warmup_ratio = 0.06,
    label_smooth = 0.05,          # re-enable: RoBERTa is stable enough from epoch 1
    grad_clip    = 1.0,
    seed         = 42,
    val_size     = 0.10,
    max_cpu_rows = 5_000,
    save_dir     = "./roberta_sarcasm_model",
)
BATCH      = CFG["batch_gpu"] if device.type == "cuda" else CFG["batch_cpu"]
GRAD_ACCUM = CFG.get("grad_accum", 1) if device.type == "cuda" else 1

# RoBERTa: fp16 works perfectly and is fastest on T4.
# DeBERTa-v3: needs bf16 (fp16 raises "unscale FP16 gradients" error).
if device.type == "cuda":
    IS_DEBERTA = "deberta" in CFG["model_name"].lower()
    USE_FP16   = not IS_DEBERTA                      # RoBERTa → fp16
    USE_BF16   = IS_DEBERTA and torch.cuda.is_bf16_supported()  # DeBERTa → bf16
else:
    USE_FP16 = False
    USE_BF16 = False

print(f"⚙️  Config: model={CFG['model_name']}  "
      f"batch={BATCH}×{GRAD_ACCUM}={BATCH*GRAD_ACCUM}(eff)  epochs={CFG['epochs']}")
_prec_str = "fp16" if USE_FP16 else ("bf16" if USE_BF16 else "fp32")
print(f"   Precision: {_prec_str}")


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  TEXT CLEANING
# ═══════════════════════════════════════════════════════════════════════════════
_URL_RE   = re.compile(r"https?://\S+|www\.\S+")
_MD_RE    = re.compile(r"[*_~`>|#]")
_WS_RE    = re.compile(r"\s{2,}")

def clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = _URL_RE.sub("", text)
    text = _MD_RE.sub("", text)
    text = text.replace("\n", " ").replace("\r", " ")
    text = _WS_RE.sub(" ", text).strip()
    return text[:512]   # hard cap before tokenizer


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  LOAD DATASETS
# ═══════════════════════════════════════════════════════════════════════════════
from datasets import load_dataset as hf_load

frames = []

# ── 5a. Reddit SARC — from Google Drive zip  ──────────────────────────────────
# Reads danofer-sarcasm.zip directly from your Google Drive (no API key needed).
# The zip must be in the root of My Drive: /content/drive/MyDrive/danofer-sarcasm.zip
DRIVE_ZIP  = "/content/drive/MyDrive/danofer-sarcasm.zip"
EXTRACT_TO = "/tmp/sarcasm_kaggle"
KAGGLE_LOADED = False

print("\n📥  Loading Reddit SARC from Google Drive zip…")
try:
    import zipfile, glob

    # Mount Drive if not already mounted
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)

    if not os.path.exists(DRIVE_ZIP):
        raise FileNotFoundError(
            f"Zip not found at {DRIVE_ZIP}\n"
            "     → Make sure danofer-sarcasm.zip is in the ROOT of your Google Drive "
            "(not inside any folder). If it is in a subfolder, update DRIVE_ZIP above."
        )

    print(f"  📦  Extracting {DRIVE_ZIP} …")
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        zf.extractall(EXTRACT_TO)

    # Find the CSV — handles any nesting inside the zip
    csvs = glob.glob(f"{EXTRACT_TO}/**/*.csv", recursive=True)
    if not csvs:
        raise FileNotFoundError(f"No CSV found inside the zip. Contents: {os.listdir(EXTRACT_TO)}")

    # Prefer the balanced training file if multiple CSVs exist
    kaggle_csv = next((p for p in csvs if "train-balanced" in p.lower()), csvs[0])
    print(f"  📄  Reading {os.path.basename(kaggle_csv)} …")

    # Cap at 80k rows: keeps total dataset ~112k, training ~25 min on T4.
    # Reddit SARC is highly redundant at scale — 80k rows covers the full
    # vocabulary of sarcastic patterns without the 8+ hour training time.
    df_reddit = pd.read_csv(kaggle_csv, nrows=80_000)
    print(f"  📋  Columns: {list(df_reddit.columns)}")

    # Standardize column names flexibly
    col_map = {}
    for c in df_reddit.columns:
        lc = c.lower()
        if "parent" in lc:
            col_map[c] = "context"
        elif "comment" in lc or "response" in lc or ("text" in lc and "parent" not in lc):
            col_map[c] = "response"
        elif "label" in lc or "sarc" in lc:
            col_map[c] = "label"
    df_reddit.rename(columns=col_map, inplace=True)

    if "response" not in df_reddit.columns:
        raise KeyError(f"Cannot find text column. Available: {list(df_reddit.columns)}")
    if "label" not in df_reddit.columns:
        raise KeyError(f"Cannot find label column. Available: {list(df_reddit.columns)}")
    if "context" not in df_reddit.columns:
        df_reddit["context"] = ""

    df_reddit = df_reddit[["label", "response", "context"]].dropna(subset=["response"])
    df_reddit["label"]    = df_reddit["label"].astype(int)
    df_reddit["response"] = df_reddit["response"].apply(clean)
    df_reddit["context"]  = df_reddit["context"].apply(clean)
    frames.append(df_reddit)

    has_ctx = (df_reddit["context"] != "").sum()
    print(f"  ✅ Reddit SARC (Drive)  →  {len(df_reddit):,} rows  "
          f"| context pairs: {has_ctx:,} "
          f"({'yes' if has_ctx > 0 else 'none — no parent_comment column in this CSV'})")
    KAGGLE_LOADED = True

except Exception as e:
    print(f"  ⚠️  Drive zip failed ({type(e).__name__}): {e}")
    print("     Falling back to public mirrors…")

# ── 5b. Reddit SARC — public mirror fallback (only if Drive zip fails)  ───────
if not KAGGLE_LOADED:
    REDDIT_MIRRORS = [
        # HuggingFace-hosted copies (more reliable than raw GitHub)
        "https://huggingface.co/datasets/MidhunKanadan/sarcasm_reddit/resolve/main/train-balanced-sarcasm.csv",
        "https://huggingface.co/datasets/rohitsatish/reddit_sarcasm/resolve/main/train-balanced-sarcasm.csv",
        # Original GitHub (may 404)
        "https://raw.githubusercontent.com/AniSkywalker/SarcasmDetection/main/resource/train-balanced-sarcasm.csv",
        "https://raw.githubusercontent.com/AniSkywalker/SarcasmDetection/master/resource/train-balanced-sarcasm.csv",
    ]
    for url in REDDIT_MIRRORS:
        try:
            df_reddit = pd.read_csv(url, nrows=80_000)
            if "parent_comment" in df_reddit.columns:
                df_reddit = df_reddit[["label","comment","parent_comment"]].copy()
                df_reddit.columns = ["label","response","context"]
            else:
                tc = next(c for c in df_reddit.columns if "text" in c.lower() or "comment" in c.lower())
                lc = next(c for c in df_reddit.columns if "label" in c.lower() or "sarc" in c.lower())
                df_reddit = pd.DataFrame({"label": df_reddit[lc], "response": df_reddit[tc], "context": ""})
            df_reddit["response"] = df_reddit["response"].apply(clean)
            df_reddit["context"]  = df_reddit["context"].apply(clean)
            df_reddit["label"]    = df_reddit["label"].astype(int)
            frames.append(df_reddit)
            print(f"  ✅ Mirror Reddit SARC  →  {len(df_reddit):,} rows")
            break
        except Exception as e:
            print(f"  ⚠️  {url[:60]}…: {e}")

# ── 5c. TweetEval Irony  ─────────────────────────────────────────────────────
print("\n📥  Loading TweetEval Irony…")
try:
    ds_tweet = hf_load("tweet_eval", "irony", split="train+test", trust_remote_code=False)
    df_tweet = pd.DataFrame({
        "response": [clean(t) for t in ds_tweet["text"]],
        "label":    ds_tweet["label"],
        "context":  "",
    })
    frames.append(df_tweet)
    print(f"  ✅ TweetEval  →  {len(df_tweet):,} rows")
except Exception as e:
    print(f"  ⚠️  TweetEval: {e}")

# ── 5d. News Headlines  ──────────────────────────────────────────────────────
print("\n📥  Loading Sarcasm Headlines…")
HL_SOURCES = [("raquiba","Sarcasm_News_Headline"), ("jakehobbs","sarcasm")]
for owner, name in HL_SOURCES:
    try:
        ds_hl    = hf_load(f"{owner}/{name}", split="train", trust_remote_code=False)
        txt_col  = next(c for c in ds_hl.column_names if "headline" in c.lower() or "text" in c.lower())
        lbl_col  = next(c for c in ds_hl.column_names if "sarc" in c.lower() or "label" in c.lower())
        df_hl    = pd.DataFrame({
            "response": [clean(t) for t in ds_hl[txt_col]],
            "label":    [int(l) for l in ds_hl[lbl_col]],
            "context":  "",
        })
        frames.append(df_hl)
        print(f"  ✅ {owner}/{name}  →  {len(df_hl):,} rows")
        break
    except Exception as e:
        print(f"  ⚠️  {owner}/{name}: {e}")

# ── 5e. Additional HuggingFace sarcasm datasets  ─────────────────────────────
print("\n📥  Loading additional HuggingFace sarcasm datasets…")
HF_EXTRAS = [
    # (dataset_id, text_col_hint, label_col_hint, split)
    ("mattlaux/nlp_sarcasm",              "text",     "label",        "train"),
    ("EleutherAI/twitter_sentiment",      "text",     "label",        "train"),
    ("UnfilteredAI/sarcasm-data",         "text",     "label",        "train"),
    ("Litify/sarcasm_detection",          "text",     "label",        "train"),
]
for ds_id, tcol, lcol, split in HF_EXTRAS:
    try:
        ds_ex = hf_load(ds_id, split=split, trust_remote_code=False)
        tc = next((c for c in ds_ex.column_names if tcol in c.lower()), None)
        lc = next((c for c in ds_ex.column_names if lcol in c.lower()), None)
        if tc is None or lc is None:
            raise ValueError(f"cols not found. Available: {ds_ex.column_names}")
        df_ex = pd.DataFrame({
            "response": [clean(t) for t in ds_ex[tc]],
            "label":    [int(l) for l in ds_ex[lc]],
            "context":  "",
        })
        # Only keep if binary labels 0/1
        unique_labels = set(df_ex["label"].unique())
        if not unique_labels.issubset({0, 1}):
            raise ValueError(f"Non-binary labels: {unique_labels}")
        frames.append(df_ex)
        print(f"  ✅ {ds_id}  →  {len(df_ex):,} rows")
    except Exception as e:
        print(f"  ⚠️  {ds_id}: {e}")

# ── 5f. Merge + balance  ─────────────────────────────────────────────────────
if not frames:
    raise RuntimeError("❌ No datasets loaded — check internet access.")

combined = pd.concat(frames, ignore_index=True)
combined = combined.dropna(subset=["response"])
combined = combined[combined["response"].str.strip() != ""]
combined["context"]  = combined["context"].fillna("").astype(str)
combined["response"] = combined["response"].astype(str)
combined["label"]    = combined["label"].astype(int)

# CPU guard
if device.type == "cpu":
    MAX = CFG["max_cpu_rows"]
    print(f"\n⚠️  CPU mode — trimming to {MAX*2:,} samples. Use T4 GPU for full training.")
    combined = (combined.groupby("label", group_keys=False)
                        .apply(lambda g: g.sample(min(MAX, len(g)), random_state=CFG["seed"]))
                        .reset_index(drop=True))

# Balance
min_cls  = combined["label"].value_counts().min()
combined = (combined.groupby("label", group_keys=False)
                    .apply(lambda g: g.sample(min(min_cls, len(g)), random_state=CFG["seed"]))
                    .reset_index(drop=True)
                    .sample(frac=1, random_state=CFG["seed"])
                    .reset_index(drop=True))

print(f"\n📊 Dataset: {len(combined):,} balanced samples")
print(f"   Sarcastic  : {combined['label'].sum():,}")
print(f"   Normal     : {(combined['label']==0).sum():,}")
print(f"   Has context: {(combined['context']!='').sum():,}")
SOURCES_USED = [f"{len(f):,} rows" for f in frames]


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  TOKENIZER  (DeBERTa-v3 uses SentencePiece — load from hub)
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n🔤  Loading tokenizer ({CFG['model_name']})…")
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
print(f"✅  Tokenizer ready  |  vocab_size={tokenizer.vocab_size:,}")


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  ENCODING — left-truncated context, right-truncated response
# ═══════════════════════════════════════════════════════════════════════════════
def encode_pairs(ctxs: list, ress: list):
    """
    Context-aware encoding for DeBERTa (fast tokenizer rejects None).
    - If ALL contexts are empty, encode responses as single sequences.
    - Otherwise encode as pairs with truncation="longest_first":
        * truncates whichever of the two sequences is currently longer
        * robust to very short / empty contexts (avoids the "too short
          to truncate" error that "only_first" raises on short pairs)
    """
    has_any_ctx = any(c and c.strip() for c in ctxs)
    if has_any_ctx:
        text_a = [c.strip() if (c and c.strip()) else "" for c in ctxs]
        return tokenizer(
            text=text_a,
            text_pair=[r for r in ress],
            truncation="longest_first",
            padding=True,
            max_length=CFG["max_len"],
            return_tensors=None,
        )
    else:
        return tokenizer(
            text=[r for r in ress],
            truncation=True,
            padding=True,
            max_length=CFG["max_len"],
            return_tensors=None,
        )

# Train/val split
(ctx_tr, ctx_val,
 res_tr, res_val,
 y_tr,   y_val) = train_test_split(
    combined["context"].tolist(),
    combined["response"].tolist(),
    combined["label"].tolist(),
    test_size=CFG["val_size"], random_state=CFG["seed"],
    stratify=combined["label"].tolist(),
)
print(f"   Train: {len(y_tr):,}  |  Val: {len(y_val):,}")

print("   Tokenizing train…")
train_enc = encode_pairs(ctx_tr, res_tr)
print("   Tokenizing val…")
val_enc   = encode_pairs(ctx_val, res_val)
print("✅  Tokenization done!")


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  DATASET CLASS  (with label smoothing built in)
# ═══════════════════════════════════════════════════════════════════════════════
class SarcasmDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels, smooth=0.0):
        self.enc    = enc
        self.labels = labels
        self.smooth = smooth   # applied at loss level instead — kept for reference

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_ds = SarcasmDataset(train_enc, y_tr)
val_ds   = SarcasmDataset(val_enc,   y_val)


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  CUSTOM TRAINER WITH LABEL SMOOTHING
# ═══════════════════════════════════════════════════════════════════════════════
class SmoothedTrainer(Trainer):
    """Injects label smoothing into the standard CE loss."""
    def __init__(self, *args, label_smoothing=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self._ls = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = CrossEntropyLoss(label_smoothing=self._ls)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ═══════════════════════════════════════════════════════════════════════════════
# 10. MODEL
# ═══════════════════════════════════════════════════════════════════════════════
print(f"\n🧠  Loading {CFG['model_name']}…")
# Suppress the verbose LOAD REPORT for non-DeBERTa models
import logging
logging.getLogger("deberta").setLevel(logging.ERROR)
# Note: do NOT pass dropout kwargs to from_pretrained for DeBERTa-v3.
# DeBERTa-v3 uses hidden_dropout / attention_dropout (not the BERT names).
# Passing unknown kwargs raises a warning and can corrupt the config object,
# which in turn causes the CUDA device-side assert on .to(device).
# Dropout is already set to sensible defaults in the pretrained config.
model = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=2,
    ignore_mismatched_sizes=True,   # suppresses the expected "MISSING classifier" warning
)
model.to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅  Model loaded  ({n_params:.0f}M parameters)")


# ═══════════════════════════════════════════════════════════════════════════════
# 11. TRAINING
# ═══════════════════════════════════════════════════════════════════════════════
def compute_metrics(pred):
    lbls  = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(lbls, preds, average="binary")
    return {
        "accuracy":  round(accuracy_score(lbls, preds), 4),
        "f1":        round(f1,  4),
        "precision": round(p,   4),
        "recall":    round(r,   4),
    }

STEPS_PER_EPOCH = len(train_ds) // BATCH
TOTAL_STEPS     = STEPS_PER_EPOCH * CFG["epochs"]
WARMUP_STEPS    = int(TOTAL_STEPS * CFG["warmup_ratio"])

training_args = TrainingArguments(
    output_dir                  = "./deberta_sarcasm_ckpt",
    num_train_epochs            = CFG["epochs"],
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    logging_strategy            = "epoch",
    learning_rate               = CFG["lr"],
    weight_decay                = CFG["weight_decay"],
    warmup_steps                = WARMUP_STEPS,
    lr_scheduler_type           = "cosine",
    max_grad_norm               = CFG["grad_clip"],
    fp16                        = USE_FP16,
    bf16                        = USE_BF16,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    report_to                   = "none",
    seed                        = CFG["seed"],
)

trainer = SmoothedTrainer(
    model             = model,
    args              = training_args,
    train_dataset     = train_ds,
    eval_dataset      = val_ds,
    compute_metrics   = compute_metrics,
    label_smoothing   = CFG["label_smooth"],
    callbacks         = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print(f"\n🚀  Training DeBERTa-v3  |  Epochs: {CFG['epochs']}  |  "
      f"Batch: {BATCH}×{GRAD_ACCUM}={BATCH*GRAD_ACCUM}(eff)  |  LR: {CFG['lr']}")
print(f"   Warmup steps: {WARMUP_STEPS}  |  Label smoothing: {CFG['label_smooth']}")
trainer.train()


# ═══════════════════════════════════════════════════════════════════════════════
# 12. EVALUATION
# ═══════════════════════════════════════════════════════════════════════════════
print("\n📊  Final Evaluation…")
eval_res  = trainer.evaluate()
val_preds = np.argmax(trainer.predict(val_ds).predictions, axis=1)
cm_matrix = confusion_matrix(y_val, val_preds)

print(f"\n{'─'*44}")
print(f"  Accuracy  : {eval_res['eval_accuracy']*100:.2f}%")
print(f"  F1 Score  : {eval_res['eval_f1']*100:.2f}%")
print(f"  Precision : {eval_res['eval_precision']*100:.2f}%")
print(f"  Recall    : {eval_res['eval_recall']*100:.2f}%")
print(f"{'─'*44}")
print(classification_report(y_val, val_preds, target_names=["Not Sarcastic","Sarcastic"]))


# ═══════════════════════════════════════════════════════════════════════════════
# 13. SAVE
# ═══════════════════════════════════════════════════════════════════════════════
model.save_pretrained(CFG["save_dir"])
tokenizer.save_pretrained(CFG["save_dir"])
print(f"\n💾  Model saved → '{CFG['save_dir']}'")
try:
    import shutil
    if os.path.exists("/content/drive/MyDrive"):
        shutil.copytree(CFG["save_dir"], "/content/drive/MyDrive/deberta_sarcasm_v2",
                        dirs_exist_ok=True)
        print("💾  Also saved to Google Drive!")
except Exception:
    pass


# ═══════════════════════════════════════════════════════════════════════════════
# 14. INFERENCE ENGINE
# ═══════════════════════════════════════════════════════════════════════════════
model.eval()
LABELS     = ["Not Sarcastic 🙂", "Sarcastic 😏"]
LABEL_COLS = ["#34d399", "#f87171"]   # green / red
_history   = []

def _proba(texts: list[str]) -> np.ndarray:
    """LIME-compatible: raw strings → (N, 2) probability array."""
    enc = tokenizer(
        texts, truncation=True, padding=True,
        max_length=CFG["max_len"], return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    return torch.softmax(logits, dim=1).cpu().numpy()


def _infer(context: str, response: str):
    """Core inference — returns (pred_idx, conf, probs[2])."""
    ctx = context.strip()
    if ctx:
        enc = tokenizer(
            ctx, response.strip(),
            truncation="longest_first", padding=True,
            max_length=CFG["max_len"], return_tensors="pt"
        )
    else:
        enc = tokenizer(
            response.strip(),
            truncation=True, padding=True,
            max_length=CFG["max_len"], return_tensors="pt"
        )
    enc = enc.to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs    = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    conf     = float(probs[pred_idx])
    return pred_idx, conf, probs


# ── Sarcasm intensity mapping ─────────────────────────────────────────────────
def _intensity(sarc_prob: float) -> tuple[str, str]:
    """Returns (label, css_color) for the intensity band."""
    if sarc_prob >= 0.90: return "🔥 Scorching Sarcasm",   "#ef4444"
    if sarc_prob >= 0.75: return "😬 High Sarcasm",         "#f97316"
    if sarc_prob >= 0.55: return "🤔 Moderate Sarcasm",     "#eab308"
    if sarc_prob >= 0.40: return "😐 Borderline",           "#a3a3a3"
    if sarc_prob >= 0.25: return "🙂 Probably Sincere",     "#4ade80"
    return                        "✅ Clearly Sincere",      "#34d399"


def predict_sarcasm(context: str, response: str):
    if not response.strip():
        warn = ("<div style='color:#fbbf24;padding:1rem;background:#1c1917;"
                "border-radius:12px;border:1px solid #fbbf24'>⚠️  Enter a message to classify.</div>")
        return warn, None, pd.DataFrame(_history)

    pred_idx, conf, probs = _infer(context, response)
    label   = LABELS[pred_idx]
    color   = LABEL_COLS[pred_idx]
    int_label, int_color = _intensity(float(probs[1]))

    # ── Result HTML  ──────────────────────────────────────────────────────────
    result_html = f"""
<div style="font-family:'DM Mono',monospace;padding:1.8rem;border-radius:20px;
            background:#111827;border:2px solid {color};position:relative;overflow:hidden">
  <!-- subtle grid background -->
  <div style="position:absolute;inset:0;
              background:repeating-linear-gradient(0deg,transparent,transparent 28px,
              rgba(255,255,255,0.02) 28px,rgba(255,255,255,0.02) 29px),
              repeating-linear-gradient(90deg,transparent,transparent 28px,
              rgba(255,255,255,0.02) 28px,rgba(255,255,255,0.02) 29px);
              pointer-events:none"></div>

  <!-- verdict -->
  <div style="font-size:2.4rem;font-weight:900;color:{color};letter-spacing:-.03em;
              text-shadow:0 0 40px {color}55">{label}</div>

  <!-- intensity badge -->
  <div style="display:inline-block;margin-top:.5rem;padding:.3rem .9rem;
              background:{int_color}22;border:1px solid {int_color}77;
              border-radius:999px;color:{int_color};font-size:.85rem;font-weight:600">
    {int_label}
  </div>

  <!-- probability bars -->
  <div style="margin:1.2rem 0 .5rem;font-size:.75rem;color:#6b7280;font-weight:700;
              letter-spacing:.1em;text-transform:uppercase">Confidence breakdown</div>

  <!-- Sarcastic bar -->
  <div style="display:flex;align-items:center;gap:.8rem;margin-bottom:.5rem">
    <div style="width:90px;color:#9ca3af;font-size:.82rem;flex-shrink:0">😏 Sarcastic</div>
    <div style="flex:1;background:#1f2937;border-radius:6px;height:10px;overflow:hidden">
      <div style="width:{probs[1]*100:.1f}%;background:#f87171;height:100%;border-radius:6px;
                  transition:width .5s cubic-bezier(.4,0,.2,1)"></div>
    </div>
    <div style="width:48px;text-align:right;color:#f87171;font-size:.9rem;font-weight:700">
      {probs[1]*100:.1f}%
    </div>
  </div>

  <!-- Not-Sarcastic bar -->
  <div style="display:flex;align-items:center;gap:.8rem">
    <div style="width:90px;color:#9ca3af;font-size:.82rem;flex-shrink:0">🙂 Sincere</div>
    <div style="flex:1;background:#1f2937;border-radius:6px;height:10px;overflow:hidden">
      <div style="width:{probs[0]*100:.1f}%;background:#34d399;height:100%;border-radius:6px;
                  transition:width .5s cubic-bezier(.4,0,.2,1)"></div>
    </div>
    <div style="width:48px;text-align:right;color:#34d399;font-size:.9rem;font-weight:700">
      {probs[0]*100:.1f}%
    </div>
  </div>

  <!-- input echo -->
  <div style="margin-top:1.2rem;padding:.9rem;background:#0f172a;border-radius:12px;
              font-size:.82rem;color:#6b7280;line-height:1.7">
    <span style="color:#7c3aed;font-weight:700">CTX :</span>
    {context.strip() if context.strip() else '<i>none</i>'}<br>
    <span style="color:#7c3aed;font-weight:700">MSG :</span> {response.strip()}
  </div>
</div>"""

    # ── LIME explanation  ─────────────────────────────────────────────────────
    lime_fig = None
    try:
        combined_text = (
            f"{context.strip()} {tokenizer.sep_token} {response.strip()}"
            if context.strip() else response.strip()
        )
        explainer = LimeTextExplainer(class_names=["Sincere", "Sarcastic"])
        exp = explainer.explain_instance(
            combined_text, _proba,
            num_features=14, num_samples=400, labels=[1],
        )
        words_w = exp.as_list(label=1)
        words, weights = zip(*words_w)
        max_abs = max(abs(w) for w in weights) or 1

        bar_colors = [
            f"rgba(248,113,113,{min(1, abs(w)/max_abs*0.9+0.1):.2f})"
            if w > 0 else
            f"rgba(52,211,153,{min(1, abs(w)/max_abs*0.9+0.1):.2f})"
            for w in weights
        ]

        lime_fig = go.Figure(go.Bar(
            x=list(weights),
            y=list(words),
            orientation="h",
            marker=dict(color=bar_colors, line=dict(width=0)),
            text=[f"{w:+.3f}" for w in weights],
            textposition="outside",
            textfont=dict(family="DM Mono", size=11, color="#94a3b8"),
        ))
        lime_fig.update_layout(
            title=dict(
                text="<b>Word-Level Attribution</b>  "
                     "<span style='font-size:11px;color:#6b7280'>"
                     "🔴 → Sarcastic &nbsp;·&nbsp; 🟢 → Sincere</span>",
                font=dict(family="DM Mono", size=14, color="#e2e8f0"),
            ),
            height=420,
            plot_bgcolor="#0f172a",
            paper_bgcolor="#111827",
            font=dict(family="DM Mono", size=11, color="#94a3b8"),
            xaxis=dict(
                zeroline=True, zerolinecolor="#374151", zerolinewidth=2,
                gridcolor="#1f2937", tickfont=dict(color="#6b7280"),
            ),
            yaxis=dict(autorange="reversed", tickfont=dict(color="#e2e8f0")),
            margin=dict(l=10, r=50, t=55, b=10),
        )
    except Exception:
        pass  # LIME is optional

    # ── History update  ───────────────────────────────────────────────────────
    _history.append({
        "Context":    (context[:55] + "…") if len(context) > 55 else (context or "—"),
        "Message":    (response[:65] + "…") if len(response) > 65 else response,
        "Prediction": label,
        "Confidence": f"{conf*100:.1f}%",
        "Sarc %":     f"{probs[1]*100:.1f}%",
    })

    return result_html, lime_fig, pd.DataFrame(_history)


def batch_predict(context: str, messages: str):
    rows = [m.strip() for m in messages.splitlines() if m.strip()]
    if not rows:
        return pd.DataFrame()

    ctx = context.strip()
    if ctx:
        enc = tokenizer(
            [ctx] * len(rows), rows,
            truncation="longest_first", padding=True,
            max_length=CFG["max_len"], return_tensors="pt"
        )
    else:
        enc = tokenizer(
            rows,
            truncation=True, padding=True,
            max_length=CFG["max_len"], return_tensors="pt"
        )
    enc = enc.to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs_all = torch.softmax(logits, dim=1).cpu().numpy()

    records = []
    for msg, probs in zip(rows, probs_all):
        idx   = int(np.argmax(probs))
        i_lbl, _ = _intensity(float(probs[1]))
        records.append({
            "Message":      msg,
            "Verdict":      LABELS[idx],
            "Intensity":    i_lbl,
            "Sarc %":       f"{probs[1]*100:.1f}%",
            "Sincere %":    f"{probs[0]*100:.1f}%",
            "Confidence":   f"{probs[idx]*100:.1f}%",
        })
    return pd.DataFrame(records)


def export_history():
    if not _history:
        return None
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=list(_history[0].keys()))
    writer.writeheader()
    writer.writerows(_history)
    path = "/tmp/sarcasm_predictions.csv"
    with open(path, "w") as f:
        f.write(buf.getvalue())
    return path


# ═══════════════════════════════════════════════════════════════════════════════
# 15. GRADIO UI  — editorial dark / monospace aesthetic
# ═══════════════════════════════════════════════════════════════════════════════
CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Mono:ital,wght@0,300;0,400;0,500;1,400&family=Syne:wght@700;800;900&display=swap');

*, *::before, *::after { box-sizing: border-box; }
html, body { background: #080c14 !important; }
.gradio-container {
  background: #080c14 !important;
  max-width: 1500px !important;
  font-family: 'DM Mono', monospace !important;
}
#MainMenu, footer, header { visibility: hidden; }

/* ── Buttons ── */
.gr-button, button.lg {
  font-family: 'DM Mono', monospace !important;
  font-weight: 500 !important;
  background: #7c3aed !important;
  color: #fff !important;
  border: 1px solid #6d28d9 !important;
  border-radius: 8px !important;
  letter-spacing: .03em !important;
  transition: all .2s !important;
}
.gr-button:hover, button.lg:hover {
  background: #6d28d9 !important;
  transform: translateY(-1px) !important;
  box-shadow: 0 6px 24px rgba(124,58,237,0.4) !important;
}

/* ── Textboxes ── */
textarea, input[type=text] {
  font-family: 'DM Mono', monospace !important;
  background: #0f172a !important;
  color: #e2e8f0 !important;
  border: 1px solid #1e293b !important;
  border-radius: 10px !important;
  caret-color: #7c3aed;
}
textarea:focus, input[type=text]:focus {
  border-color: #7c3aed !important;
  box-shadow: 0 0 0 3px rgba(124,58,237,0.15) !important;
}

/* ── Labels ── */
label, .gr-form label { color: #6b7280 !important; font-size:.78rem !important; letter-spacing:.08em !important; text-transform:uppercase !important; font-weight:500 !important; }

/* ── Tabs ── */
.tab-nav { background: transparent !important; border-bottom: 1px solid #1e293b !important; }
.tab-nav button { color: #6b7280 !important; font-family: 'DM Mono', monospace !important; font-size:.85rem !important; }
.tab-nav button.selected { color: #a78bfa !important; border-bottom: 2px solid #7c3aed !important; }

/* ── Dataframe ── */
table { background: #0f172a !important; color: #cbd5e1 !important; font-size:.8rem !important; }
th { background: #1e293b !important; color: #a78bfa !important; font-weight:600 !important; }
tr:nth-child(even) { background: #111827 !important; }

/* ── Cards ── */
.block { background: #0f172a !important; border: 1px solid #1e293b !important; border-radius: 16px !important; }
"""

HEADER_HTML = f"""
<div style="padding:2rem 0 1.5rem;border-bottom:1px solid #1e293b;margin-bottom:1.5rem">
  <div style="font-family:'Syne',sans-serif;font-size:2.8rem;font-weight:900;
              color:#f8fafc;letter-spacing:-.04em;line-height:1">
    😏 sarcasm<span style="color:#7c3aed">_</span>detector
  </div>
  <div style="margin-top:.5rem;font-family:'DM Mono',monospace;font-size:.85rem;color:#4b5563">
    DeBERTa-v3-base &nbsp;·&nbsp; Reddit SARC + TweetEval + Headlines
    &nbsp;·&nbsp; LIME Explanations
    &nbsp;·&nbsp; <span style="color:#34d399">accuracy {eval_res['eval_accuracy']*100:.1f}%</span>
    &nbsp;·&nbsp; <span style="color:#f87171">f1 {eval_res['eval_f1']*100:.1f}%</span>
  </div>
  <div style="margin-top:.8rem;font-family:'DM Mono',monospace;font-size:.78rem;color:#374151">
    Datasets: {' + '.join(SOURCES_USED)} &nbsp;·&nbsp; {len(combined):,} balanced samples
  </div>
</div>
"""

EXAMPLES = [
    ["How was the surgery?",       "Oh perfect, I only lost feeling in my left hand."],
    ["Did you enjoy the concert?", "Every off-key note was a gift to my ears."],
    ["How was traffic today?",     "Two hours for 10 km. Really efficient city planning."],
    ["Did the meeting go well?",   "Yes the client was thrilled and signed immediately."],
    ["",                           "Scientists say sleeping more is the best productivity hack."],
    ["",                           "I absolutely love Mondays. Most energising day of the week."],
    ["Are you ready for the exam?", "Never been more prepared in my life — it's not like I haven't slept."],
    ["",                           "The project is finally done and I'm genuinely happy with it."],
]

with gr.Blocks(theme=gr.themes.Base(), css=CSS, title="😏 Sarcasm Detector") as demo:

    gr.HTML(HEADER_HTML)

    with gr.Tabs():

        # ── Tab 1: Single Prediction ─────────────────────────────────────────
        with gr.Tab("/ predict"):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=380):
                    ctx_box  = gr.Textbox(
                        label="Context  (previous message — optional)",
                        placeholder="e.g.  How was your day?",
                        lines=3, max_lines=6)
                    resp_box = gr.Textbox(
                        label="Message  ← classify this",
                        placeholder="e.g.  Just the best day of my entire life.",
                        lines=3, max_lines=6)
                    pred_btn = gr.Button("→ Detect Sarcasm", variant="primary", size="lg")
                    gr.Markdown(
                        "<div style='color:#374151;font-size:.75rem;margin-top:.3rem'>"
                        "💡 Context improves accuracy for conversational sarcasm</div>")
                    gr.Examples(
                        examples=EXAMPLES, inputs=[ctx_box, resp_box],
                        label="Quick examples", examples_per_page=8)

                with gr.Column(scale=1, min_width=380):
                    result_html = gr.HTML(
                        "<div style='color:#374151;font-size:.85rem;padding:2rem;text-align:center;"
                        "border:1px dashed #1e293b;border-radius:16px'>↑ Enter a message and click detect</div>")
                    lime_plot = gr.Plot()

            gr.Markdown("---")
            with gr.Row():
                gr.Markdown("#### 📋 Prediction History")
                export_btn = gr.Button("↓ Export CSV", size="sm")
            hist_table  = gr.Dataframe(
                label="", wrap=True,
                headers=["Context","Message","Prediction","Confidence","Sarc %"])
            export_file = gr.File(label="Download", visible=False)

            pred_btn.click(
                fn=predict_sarcasm,
                inputs=[ctx_box, resp_box],
                outputs=[result_html, lime_plot, hist_table])
            export_btn.click(
                fn=export_history,
                outputs=[export_file])
            export_file.change(
                fn=lambda f: gr.update(visible=f is not None),
                inputs=[export_file],
                outputs=[export_file])

        # ── Tab 2: Batch ─────────────────────────────────────────────────────
        with gr.Tab("/ batch"):
            gr.Markdown(
                "### Batch Classification\n"
                "One message per line. Optional shared context applied to all.")
            batch_ctx  = gr.Textbox(
                label="Shared context (optional)", lines=2,
                placeholder="e.g.  How was the holiday?")
            batch_msgs = gr.Textbox(
                label="Messages — one per line", lines=9,
                placeholder="Oh just absolutely wonderful\nThe beach was nice\nSure, love losing luggage\nFlight was on time for once")
            batch_btn  = gr.Button("→ Classify All", variant="primary")
            batch_out  = gr.Dataframe(label="Results", wrap=True)
            batch_btn.click(fn=batch_predict, inputs=[batch_ctx, batch_msgs], outputs=batch_out)

        # ── Tab 3: Model Stats ────────────────────────────────────────────────
        with gr.Tab("/ model_stats"):
            acc  = eval_res["eval_accuracy"] * 100
            f1   = eval_res["eval_f1"]        * 100
            prec = eval_res["eval_precision"] * 100
            rec  = eval_res["eval_recall"]    * 100

            # Confusion matrix
            cm_fig = go.Figure(go.Heatmap(
                z=cm_matrix.tolist(),
                x=["Predicted: Sincere","Predicted: Sarcastic"],
                y=["Actual: Sincere","Actual: Sarcastic"],
                colorscale=[[0,"#0f172a"],[0.5,"#7c3aed"],[1,"#f87171"]],
                text=[[str(v) for v in row] for row in cm_matrix.tolist()],
                texttemplate="<b>%{text}</b>", showscale=True,
            ))
            cm_fig.update_layout(
                title=dict(text="<b>Confusion Matrix</b>",
                           font=dict(family="DM Mono", color="#e2e8f0")),
                height=420,
                plot_bgcolor="#0f172a", paper_bgcolor="#0f172a",
                font=dict(family="DM Mono", color="#94a3b8"),
                xaxis=dict(tickfont=dict(color="#94a3b8")),
                yaxis=dict(tickfont=dict(color="#94a3b8")),
                margin=dict(l=10,r=10,t=50,b=10),
            )

            # Radar chart
            radar_fig = go.Figure(go.Scatterpolar(
                r=[acc, f1, prec, rec, acc],
                theta=["Accuracy","F1","Precision","Recall","Accuracy"],
                fill="toself",
                fillcolor="rgba(124,58,237,0.2)",
                line=dict(color="#7c3aed", width=2),
                marker=dict(color="#a78bfa", size=8),
            ))
            radar_fig.update_layout(
                polar=dict(
                    bgcolor="#0f172a",
                    radialaxis=dict(range=[0,100], tickfont=dict(color="#6b7280"),
                                    gridcolor="#1e293b"),
                    angularaxis=dict(tickfont=dict(color="#e2e8f0"),
                                     gridcolor="#1e293b"),
                ),
                paper_bgcolor="#0f172a",
                font=dict(family="DM Mono", color="#94a3b8"),
                height=380,
                margin=dict(l=40,r=40,t=20,b=20),
            )

            # ── Build HTML fragments as plain strings first (avoids nested f-string
            #    triple-quote tokenizer crash on Python < 3.12)
            _metric_cards = ""
            for _nm, _val, _c, _ico in [
                ("Accuracy",  acc,  "#a78bfa", "🎯"),
                ("F1 Score",  f1,   "#f87171", "⚡"),
                ("Precision", prec, "#34d399", "🔍"),
                ("Recall",    rec,  "#38bdf8", "📡"),
            ]:
                _metric_cards += (
                    '<div style="background:#0f172a;border:1px solid #1e293b;'
                    'border-top:3px solid ' + _c + ';border-radius:14px;'
                    'padding:1.4rem;text-align:center">'
                    '<div style="font-size:1.8rem">' + _ico + '</div>'
                    '<div style="font-size:2rem;font-weight:700;color:' + _c + ';'
                    'letter-spacing:-.03em;margin:.3rem 0">'
                    + f"{_val:.1f}" + '<span style="font-size:1rem">%</span></div>'
                    '<div style="color:#4b5563;font-size:.7rem;letter-spacing:.1em;'
                    'text-transform:uppercase">' + _nm + '</div>'
                    '</div>'
                )

            _arch_rows = ""
            for _k, _v in [
                ("Base Model",       f"DeBERTa-v3-base  ({n_params:.0f}M params)"),
                ("Training",         f"{CFG['epochs']} epochs · LR={CFG['lr']} · cosine · batch={BATCH}"),
                ("Regularisation",   f"label_smooth={CFG['label_smooth']} · weight_decay={CFG['weight_decay']} · grad_clip={CFG['grad_clip']}"),
                ("Early Stopping",   "patience=2 epochs on eval F1"),
                ("Datasets",         " + ".join(SOURCES_USED)),
                ("Total Samples",    f"{len(combined):,} (balanced)"),
                ("Context Strategy", "text_a=parent_comment · truncation='longest_first'"),
                ("Explainability",   "LIME  (14 top features, 400 perturbation samples)"),
                ("Device",           str(device).upper()),
            ]:
                _arch_rows += (
                    '<tr>'
                    '<td style="padding:.45rem .8rem;color:#4b5563;width:32%;font-size:.78rem;'
                    'border-bottom:1px solid #111827">' + _k + '</td>'
                    '<td style="padding:.45rem .8rem;color:#e2e8f0;font-size:.78rem;'
                    'border-bottom:1px solid #111827">' + _v + '</td>'
                    '</tr>'
                )

            stats_html = (
                '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:1rem;'
                'margin:1.5rem 0;font-family:\'DM Mono\',monospace">'
                + _metric_cards +
                '</div>'
                '<div style="font-family:\'DM Mono\',monospace;background:#0f172a;'
                'border:1px solid #1e293b;border-radius:14px;padding:1.5rem;margin-top:1rem">'
                '<div style="color:#7c3aed;font-size:.7rem;letter-spacing:.12em;font-weight:600;'
                'text-transform:uppercase;margin-bottom:1rem">Architecture</div>'
                '<table style="width:100%;border-collapse:collapse">'
                + _arch_rows +
                '</table></div>'
            )

            gr.HTML(stats_html)
            with gr.Row():
                gr.Plot(value=cm_fig)
                gr.Plot(value=radar_fig)


# ═══════════════════════════════════════════════════════════════════════════════
# 16. LAUNCH
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("🚀  LAUNCHING  sarcasm_detector v2.0  (DeBERTa-v3)")
print("="*70)
demo.launch(share=True, debug=False)
print("✅  Dashboard LIVE")

✅ Imports ready  |  Device: cuda
✅ GPU ready  |  Tesla T4  |  15.6 GB total
⚙️  Config: model=roberta-base  batch=32×1=32(eff)  epochs=3
   Precision: fp16

📥  Loading Reddit SARC from Google Drive zip…
  📦  Extracting /content/drive/MyDrive/danofer-sarcasm.zip …
  📄  Reading train-balanced-sarcasm.csv …
  📋  Columns: ['label', 'comment', 'author', 'subreddit', 'score', 'ups', 'downs', 'date', 'created_utc', 'parent_comment']
  ✅ Reddit SARC (Drive)  →  79,997 rows  | context pairs: 79,993 (yes)

📥  Loading TweetEval Irony…


  ✅ TweetEval  →  3,646 rows

📥  Loading Sarcasm Headlines…


Repo card metadata block was not found. Setting CardData to empty.


  ✅ raquiba/Sarcasm_News_Headline  →  28,619 rows

📥  Loading additional HuggingFace sarcasm datasets…
  ⚠️  mattlaux/nlp_sarcasm: Dataset 'mattlaux/nlp_sarcasm' doesn't exist on the Hub or cannot be accessed.
  ⚠️  EleutherAI/twitter_sentiment: Dataset 'EleutherAI/twitter_sentiment' doesn't exist on the Hub or cannot be accessed.
  ⚠️  UnfilteredAI/sarcasm-data: Dataset 'UnfilteredAI/sarcasm-data' doesn't exist on the Hub or cannot be accessed.
  ⚠️  Litify/sarcasm_detection: Dataset 'Litify/sarcasm_detection' doesn't exist on the Hub or cannot be accessed.

📊 Dataset: 99,542 balanced samples
   Sarcastic  : 49,771
   Normal     : 49,771
   Has context: 70,668

🔤  Loading tokenizer (roberta-base)…


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

...


📊  Final Evaluation…



────────────────────────────────────────────
  Accuracy  : 79.85%
  F1 Score  : 79.88%
  Precision : 79.74%
  Recall    : 80.03%
────────────────────────────────────────────
               precision    recall  f1-score   support

Not Sarcastic       0.80      0.80      0.80      4978
    Sarcastic       0.80      0.80      0.80      4977

     accuracy                           0.80      9955
    macro avg       0.80      0.80      0.80      9955
 weighted avg       0.80      0.80      0.80      9955



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


💾  Model saved → './roberta_sarcasm_model'
💾  Also saved to Google Drive!

🚀  LAUNCHING  sarcasm_detector v2.0  (DeBERTa-v3)
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6dd54daa957ae248d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅  Dashboard LIVE
